In [ ]:
from typing import Union
import pathlib
import numpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.ndimage as nd
from scipy._lib._util import normalize_axis_index
import SimpleITK as sitk

import _ni_support
import _nd_image

import deeperhistreg
from deeperhistreg.dhr_input_output.dhr_loaders import pil_loader
from deeperhistreg.dhr_input_output.dhr_loaders import tiff_loader
from deeperhistreg.dhr_pipeline.registration_params import default_initial_nonrigid
from deeperhistreg.dhr_utils import warping as w

In [15]:
def _prepad_for_spline_filter(input, mode, cval):
    if mode in ['nearest', 'grid-constant']:
        npad = 12
        if mode == 'grid-constant':
            padded = numpy.pad(input, npad, mode='constant',
                               constant_values=cval)
        elif mode == 'nearest':
            padded = numpy.pad(input, npad, mode='edge')
    else:
        # other modes have exact boundary conditions implemented so
        # no prepadding is needed
        npad = 0
        padded = input
    return padded, npad

def spline_filter1d(input, order=3, axis=-1, output=numpy.float64,
                    mode='mirror'):
    if order < 0 or order > 5:
        raise RuntimeError('spline order not supported')
    input = numpy.asarray(input)
    complex_output = numpy.iscomplexobj(input)
    output = _ni_support._get_output(output, input,
                                     complex_output=complex_output)
    if complex_output:
        spline_filter1d(input.real, order, axis, output.real, mode)
        spline_filter1d(input.imag, order, axis, output.imag, mode)
        return output
    if order in [0, 1]:
        output[...] = numpy.array(input)
    else:
        mode = _ni_support._extend_mode_to_code(mode)
        axis = normalize_axis_index(axis, input.ndim)
        _nd_image.spline_filter1d(input, order, axis, output, mode)
    return output

def spline_filter(input, order=3, output=numpy.float64, mode='mirror'):
    if order < 2 or order > 5:
        raise RuntimeError('spline order not supported')
    input = numpy.asarray(input)
    complex_output = numpy.iscomplexobj(input)
    output = _ni_support._get_output(output, input,
                                     complex_output=complex_output)
    if complex_output:
        spline_filter(input.real, order, output.real, mode)
        spline_filter(input.imag, order, output.imag, mode)
        return output
    if order not in [0, 1] and input.ndim > 0:
        for axis in range(input.ndim):
            spline_filter1d(input, order, axis, output=output, mode=mode)
            input = output
    else:
        output[...] = input[...]
    return output

def map_coordinates(input, coordinates, output=None, order=3,
                    mode='constant', cval=0.0, prefilter=True):
    if order < 0 or order > 5:
        raise RuntimeError('spline order not supported')
    input = numpy.asarray(input)
    # input = sitk.GetArrayFromImage(input)
    coordinates = numpy.asarray(coordinates)
    if numpy.iscomplexobj(coordinates):
        raise TypeError('Complex type not supported')
    output_shape = coordinates.shape[1:]
    if input.ndim < 1 or len(output_shape) < 1:
        raise RuntimeError('input and output rank must be > 0')
    print("coordinate:", coordinates)
    print(coordinates.shape[0])
    print("input:", input)
    print(input.ndim)
    if coordinates.shape[0] != input.ndim:
        raise RuntimeError('invalid shape for coordinate array')
    complex_output = numpy.iscomplexobj(input)
    output = _ni_support._get_output(output, input, shape=output_shape,
                                     complex_output=complex_output)
    if complex_output:
        kwargs = dict(order=order, mode=mode, prefilter=prefilter)
        map_coordinates(input.real, coordinates, output=output.real,
                        cval=numpy.real(cval), **kwargs)
        map_coordinates(input.imag, coordinates, output=output.imag,
                        cval=numpy.imag(cval), **kwargs)
        return output
    if prefilter and order > 1:
        padded, npad = _prepad_for_spline_filter(input, mode, cval)
        filtered = spline_filter(padded, order, output=numpy.float64,
                                 mode=mode)
    else:
        npad = 0
        filtered = input
    mode = _ni_support._extend_mode_to_code(mode)
    _nd_image.geometric_transform(filtered, None, coordinates, None, None,
                                  output, order, mode, cval, npad, None, None)
    return output

In [16]:
def warp_landmarks_w(landmarks : np.ndarray, displacement_field : np.ndarray) -> np.ndarray:
    landmarks_x = landmarks[:, 0]
    landmarks_y = landmarks[:, 1]
    ux = map_coordinates(displacement_field[0, :, :], [landmarks_y, landmarks_x], mode='nearest')
    uy = map_coordinates(displacement_field[1, :, :], [landmarks_y, landmarks_x], mode='nearest')
    new_landmarks = np.stack((landmarks_x + ux, landmarks_y + uy), axis=1)
    return new_landmarks

In [17]:
def warp_landmarks(landmarks, displacement_field, postprocessing_params):
    source_pad = postprocessing_params['pad_1']
    target_pad = postprocessing_params['pad_2']
    source_resample_ratio = postprocessing_params['source_resample_ratio']
    target_resample_ratio = postprocessing_params['target_resample_ratio']
    initial_resampling = postprocessing_params['initial_resampling']
    if initial_resampling:
        initial_resample_ratio = postprocessing_params['initial_resample_ratio']
    y_pad = target_pad[0]
    x_pad = target_pad[1]
    landmarks = landmarks * target_resample_ratio
    landmarks[:, 0] = landmarks[:, 0] + x_pad[0]
    landmarks[:, 1] = landmarks[:, 1] + y_pad[0]
    if initial_resampling:
        landmarks = landmarks / initial_resample_ratio
    print("landmarks:", landmarks)
    print("displacement_field:", displacement_field)
    warped_landmarks = warp_landmarks_w(landmarks, displacement_field)
    if initial_resampling:
        warped_landmarks = warped_landmarks * initial_resample_ratio
    y_pad = source_pad[0]
    x_pad = source_pad[1]
    warped_landmarks[:, 0] = warped_landmarks[:, 0] - x_pad[0]
    warped_landmarks[:, 1] = warped_landmarks[:, 1] - y_pad[0]
    warped_landmarks = warped_landmarks / source_resample_ratio
    return warped_landmarks

In [18]:
import SimpleITK as sitk
displacement_field = sitk.ReadImage("displacement_field.mha")
displacement_field = sitk.GetArrayFromImage(displacement_field)
displacement_field

array([[[ 15.349055 ,  14.395635 ,  14.380449 , ...,  21.591984 ,
          21.597387 ,  21.602419 ],
        [ 15.319336 ,  14.376117 ,  14.360093 , ...,  21.592358 ,
          21.597946 ,  21.60279  ],
        [ 15.289385 ,  14.366987 ,  14.349147 , ...,  21.593102 ,
          21.598598 ,  21.603443 ],
        ...,
        [-22.377434 , -22.373707 , -22.371006 , ..., -19.721058 ,
         -20.721384 , -21.721573 ],
        [-22.409388 , -22.405663 , -22.403053 , ..., -19.750263 ,
         -20.750546 , -21.750732 ],
        [-22.440039 , -22.436499 , -22.43389  , ..., -19.779564 ,
         -20.77989  , -21.780031 ]],

       [[ -7.785812 ,  -7.792683 ,  -7.7803154, ...,  31.943207 ,
          31.972614 ,  32.00202  ],
        [ -7.7729406,  -7.77972  ,  -7.767719 , ...,  30.943213 ,
          30.972666 ,  31.002028 ],
        [ -7.7517323,  -7.758237 ,  -7.7469683, ...,  29.943218 ,
          29.972672 ,  30.002033 ],
        ...,
        [-19.053717 , -19.023577 , -18.992521 , ...,  

In [19]:
import json
with open("postprocessing_params.json", 'r') as f:
    postprocessing_params = json.load(f)
postprocessing_params

{'pad_1': [[0, 0], [0, 0]],
 'pad_2': [[9, 10], [21, 21]],
 'source_resample_ratio': 0.2,
 'target_resample_ratio': 0.2,
 'initial_resampling': True,
 'initial_resample_ratio': 1}

In [22]:
landmarks = np.array([[5836,4595], [4387, 1604], [4099, 2646]])
# landmarks = np.array([[4595, 5836]])

In [23]:
warped_landmarks = warp_landmarks(landmarks, displacement_field, postprocessing_params)
# warped_landmarks = warped_landmarks * scale_factor
warped_landmarks

landmarks: [[1188.2  928. ]
 [ 898.4  329.8]
 [ 840.8  538.2]]
displacement_field: [[[ 15.349055   14.395635   14.380449  ...  21.591984   21.597387
    21.602419 ]
  [ 15.319336   14.376117   14.360093  ...  21.592358   21.597946
    21.60279  ]
  [ 15.289385   14.366987   14.349147  ...  21.593102   21.598598
    21.603443 ]
  ...
  [-22.377434  -22.373707  -22.371006  ... -19.721058  -20.721384
   -21.721573 ]
  [-22.409388  -22.405663  -22.403053  ... -19.750263  -20.750546
   -21.750732 ]
  [-22.440039  -22.436499  -22.43389   ... -19.779564  -20.77989
   -21.780031 ]]

 [[ -7.785812   -7.792683   -7.7803154 ...  31.943207   31.972614
    32.00202  ]
  [ -7.7729406  -7.77972    -7.767719  ...  30.943213   30.972666
    31.002028 ]
  [ -7.7517323  -7.758237   -7.7469683 ...  29.943218   29.972672
    30.002033 ]
  ...
  [-19.053717  -19.023577  -18.992521  ...  12.095165   12.105792
    12.111013 ]
  [-19.047945  -19.01776   -18.986658  ...  12.089256   12.099975
    12.105196 ]
  

array([[5909.98654699, 4747.25400925],
       [4578.24734879, 1718.82550621],
       [4236.11186171, 2748.5750351 ]])